In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *


# Configuration
VOLUME_BASE = "/Volumes/imdb_final_project/raw/raw_store"
ENABLE_SCHEMA_EVOLUTION = False

# Helper function to reduce code duplication
def create_bronze_table(subfolder, schema_hints):
    """Helper to create bronze ingestion logic"""
    reader = (
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "csv")
            .option("cloudFiles.schemaLocation", f"{VOLUME_BASE}/{subfolder}/_schema_checkpoint")
            .option("cloudFiles.inferColumnTypes", "true")
            .option("cloudFiles.schemaHints", schema_hints)
            .option("delimiter", "\t")
            .option("header", "true")
            .option("multiLine", "true")
            .option("escape", "\"")
            .option("nullValue", "\\N")
    )
    
    if ENABLE_SCHEMA_EVOLUTION:
        reader = reader.option("cloudFiles.schemaEvolutionMode", "addNewColumns")
    
    df = reader.load(f"{VOLUME_BASE}/{subfolder}")
    
    # # Rename columns
    # for col_name in df.columns:
    #     clean_name = col_name.replace(" ", "_").replace("#", "Number")
    #     df = df.withColumnRenamed(col_name, clean_name)
    

    import re
    for col_name in df.columns:
        # Replace invalid chars with underscore, then clean up multiple underscores
        clean_name = re.sub(r'[ ,;{}()\n\t=]+', '_', col_name)
        clean_name = clean_name.replace("#", "Number")
        # Remove leading/trailing underscores
        clean_name = clean_name.strip('_')
        df = df.withColumnRenamed(col_name, clean_name)

    # Add audit columns
    return (
        df
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
        .withColumn("ingestion_date", current_date())
    )

# ============================================
# BRONZE TABLES
# ============================================

@dlt.table(
    name="bronze_name_basics_raw",
    comment="Raw IMDB name.basics data"
)
def bronze_name_basics():
    return create_bronze_table("name_basics", "nconst STRING, birthYear STRING, deathYear STRING")

@dlt.table(
    name="bronze_title_akas_raw",
    comment="Raw IMDB title.akas data"
)
def bronze_title_akas():
    return create_bronze_table("title_akas", "titleId STRING, ordering STRING")

@dlt.table(
    name = "bronze_title_crew_raw",
    comment = "Raw IMDB title.crew data"
)
def bronze_title_crew():
    return create_bronze_table("title_crew", "tconst STRING, directors STRING, writers STRING")

@dlt.table(
    name="bronze_title_ratings_raw",
    comment="Raw IMDB title.ratings data"
)
def bronze_title_ratings():
    return create_bronze_table("title_ratings", "tconst STRING, averageRating STRING, numVotes STRING")

@dlt.table(
    name = "bronze_title_region_raw",
    comment = "Raw IMDB title.region data"
)
def bronze_title_region():
    return create_bronze_table("title_region", "REGION_CODE STRING, REGION_NAME STRING"
)
    
@dlt.table(
    name="bronze_title_episode_raw",
    comment="Raw IMDB title.episode data"
)
def bronze_title_episode():
    return create_bronze_table("title_episode", "tconst STRING, parentTconst STRING, seasonNumber STRING, episodeNumber STRING")

@dlt.table(
    name="bronze_title_principals_raw",
    comment="Raw IMDB title.principals data"
)
def bronze_title_principals():
    return create_bronze_table("title_principals", "tconst STRING, ordering STRING")

@dlt.table(
    name="bronze_title_basics_raw",
    comment="Raw IMDB title.basics data"
)
def bronze_title_basics():
    return create_bronze_table("title_basics", "tconst STRING, titleType STRING, primaryTitle STRING, originalTitle STRING")


@dlt.table(
    name = "bronze_title_language_codes_raw",
    comment = "Raw IMDB title.language_codes data"
)

def bronze_title_language_codes():
    return create_bronze_table("language_codes", "LANGUAGE_NAME STRING, LANGUAGE_CODE STRING")


In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

@dlt.table(
    name="silver.silver_name_basics",
    comment="Cleaned IMDB name basics data - preserves all records without explosion",
    table_properties={
        "quality": "silver",    
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_all_or_drop({
    "valid_nconst": "NCONST IS NOT NULL AND NCONST RLIKE '^nm[0-9]{7,}$'",
    # "valid_primary_name": "PRIMARY_NAME IS NOT NULL AND LENGTH(TRIM(PRIMARY_NAME)) > 0"
})
def silver_name_basics():
    df = dlt.read_stream("bronze_name_basics_raw")
    
    # Replace \N with NULLs
    df = df.select(
        *[when(col(c) == "\\N", None).otherwise(col(c)).alias(c) for c in df.columns]
    )
    
    # Rename columns to uppercase
    df = (
        df
        .withColumnRenamed("nconst", "NCONST")
        .withColumnRenamed("primaryName", "PRIMARY_NAME")
        .withColumnRenamed("birthYear", "BIRTH_YEAR")
        .withColumnRenamed("deathYear", "DEATH_YEAR")
        .withColumnRenamed("primaryProfession", "PRIMARY_PROFESSION")
        .withColumnRenamed("knownForTitles", "KNOWN_FOR_TITLES")
    )
    
    # Handle NULL values
    df = (
        df
        .withColumn("PRIMARY_NAME",
                   when(col("PRIMARY_NAME").isNull(), "Unknown")
                   .otherwise(col("PRIMARY_NAME")))
        .withColumn("BIRTH_YEAR", 
                   when(col("BIRTH_YEAR").isNull(), "0000")
                   .otherwise(col("BIRTH_YEAR")))
        .withColumn("DEATH_YEAR", 
                   when(col("DEATH_YEAR").isNull(), "9999")
                   .otherwise(col("DEATH_YEAR")))
        .withColumn("PRIMARY_PROFESSION", 
                   when(col("PRIMARY_PROFESSION").isNull(), "Unknown")
                   .otherwise(col("PRIMARY_PROFESSION")))
        .withColumn("KNOWN_FOR_TITLES", 
                   when(col("KNOWN_FOR_TITLES").isNull(), "Unknown")
                   .otherwise(col("KNOWN_FOR_TITLES")))
    )
    
    # Cast to integer
    df = (
        df
        .withColumn("BIRTH_YEAR", col("BIRTH_YEAR").cast("int"))
        .withColumn("DEATH_YEAR", col("DEATH_YEAR").cast("int"))
    )
    
    # Add is_alive flag
    df = df.withColumn("IS_ALIVE", when(col("DEATH_YEAR") == 9999, True).otherwise(False))
    
    # Trim whitespace
    df = (
        df
        .withColumn("PRIMARY_NAME", trim(col("PRIMARY_NAME")))
        .withColumn("PRIMARY_PROFESSION", trim(col("PRIMARY_PROFESSION")))
        .withColumn("KNOWN_FOR_TITLES", trim(col("KNOWN_FOR_TITLES")))
    )
    
    # Add silver processing timestamp
    df = df.withColumn(
        "silver_processing_timestamp", 
        current_timestamp()
    )
    
    # Select final columns
    return df.select(
        "NCONST",
        "PRIMARY_NAME", 
        "BIRTH_YEAR",
        "DEATH_YEAR",
        "IS_ALIVE",
        "PRIMARY_PROFESSION",
        "KNOWN_FOR_TITLES",
        "ingestion_timestamp",
        "silver_processing_timestamp",
        "source_file",
        "ingestion_date"
    )

In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

@dlt.table(
    name="silver.silver_title_akas",
    comment="Cleaned IMDB title.akas data with NULL handling - no explosion needed",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_all_or_drop({
    "valid_titleId": "TITLE_ID IS NOT NULL AND TITLE_ID RLIKE '^tt[0-9]{7,}$'",
    "valid_ordering": "ORDERING >= -1"
})
def silver_title_akas():
    """
    Cleans the bronze title_akas data.
    
    Transformations:
    - Replace \\N with NULLs
    - Replace NULL ordering with -1
    - Replace NULL text fields with 'Unknown'
    - Replace NULL isOriginalTitle with -1
    - Create boolean flag for isOriginalTitle
    - Trim whitespace
    """
    
    df = dlt.read_stream("bronze_title_akas_raw")
    
    # Replace \N with NULLs
    df = df.select(
        *[when(col(c) == "\\N", None).otherwise(col(c)).alias(c) for c in df.columns]
    )
    
    # Rename columns to uppercase
    df = (
        df
        .withColumnRenamed("titleId", "TITLE_ID")
        .withColumnRenamed("ordering", "ORDERING")
        .withColumnRenamed("title", "TITLE")
        .withColumnRenamed("region", "REGION")
        .withColumnRenamed("language", "LANGUAGE")
        .withColumnRenamed("types", "TYPES")
        .withColumnRenamed("attributes", "ATTRIBUTES")
        .withColumnRenamed("isOriginalTitle", "IS_ORIGINAL_TITLE")
    )
    
    # Handle NULL values
    df = (
        df
        .withColumn("ORDERING", 
                   when(col("ORDERING").isNull(), "-1")
                   .otherwise(col("ORDERING")))
        .withColumn("TITLE", 
                   when(col("TITLE").isNull(), "Unknown")
                   .otherwise(col("TITLE")))
        .withColumn("REGION", 
                   when(col("REGION").isNull(), "Unknown")
                   .otherwise(col("REGION")))
        .withColumn("LANGUAGE", 
                   when(col("LANGUAGE").isNull(), "Unknown")
                   .otherwise(col("LANGUAGE")))
        .withColumn("TYPES", 
                   when(col("TYPES").isNull(), "Unknown")
                   .otherwise(col("TYPES")))
        .withColumn("ATTRIBUTES", 
                   when(col("ATTRIBUTES").isNull(), "Unknown")
                   .otherwise(col("ATTRIBUTES")))
        .withColumn("IS_ORIGINAL_TITLE", 
                   when(col("IS_ORIGINAL_TITLE").isNull(), "-1")
                   .otherwise(col("IS_ORIGINAL_TITLE")))
    )
    
    # Cast to integer
    df = (
        df
        .withColumn("ORDERING", col("ORDERING").cast("int"))
        .withColumn("IS_ORIGINAL_TITLE", col("IS_ORIGINAL_TITLE").cast("int"))
    )
    
    # Create boolean flag for original title
    df = df.withColumn("IS_ORIGINAL_TITLE_FLAG",
                      when(col("IS_ORIGINAL_TITLE") == 1, True)
                      .when(col("IS_ORIGINAL_TITLE") == 0, False)
                      .otherwise(None))
    
    # Trim whitespace
    df = (
        df
        .withColumn("TITLE", trim(col("TITLE")))
        .withColumn("REGION", trim(col("REGION")))
        .withColumn("LANGUAGE", trim(col("LANGUAGE")))
        .withColumn("TYPES", trim(col("TYPES")))
        .withColumn("ATTRIBUTES", trim(col("ATTRIBUTES")))
    )
    
    # Add silver processing timestamp
    df = df.withColumn(
        "silver_processing_timestamp", 
        current_timestamp()
    )
    
    # Select final columns
    return df.select(
        "TITLE_ID",
        "ORDERING",
        "TITLE",
        "REGION",
        "LANGUAGE",
        "TYPES",
        "ATTRIBUTES",
        "IS_ORIGINAL_TITLE",
        "IS_ORIGINAL_TITLE_FLAG",
        "ingestion_timestamp",
        "silver_processing_timestamp",
        "source_file",
        "ingestion_date"
    )

In [0]:
@dlt.table(
    name="silver.silver_title_language_codes",
    comment="Cleaned IMDB language codes lookup table - preserves all records",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
def silver_title_language_codes():
    df = dlt.read_stream("bronze_title_language_codes_raw")
    
    # # Replace \N with NULLs
    # df = df.select(
    #     *[when(col(c) == "\\N", None).otherwise(col(c)).alias(c) for c in df.columns]
    # )
    
    # Rename columns to uppercase
    df = (
        df
        .withColumnRenamed("LANGUAGE_NAME", "LANGUAGE_NAME")
        .withColumnRenamed("LANGUAGE_CODE", "LANGUAGE_CODE")
    )
    
    # # Handle NULL values
    # df = (
    #     df
    #     .withColumn("LANGUAGE_NAME", 
    #                when(col("LANGUAGE_NAME").isNull(), "Unknown")
    #                .otherwise(col("LANGUAGE_NAME")))
    #     .withColumn("LANGUAGE_CODE", 
    #                when(col("LANGUAGE_CODE").isNull(), "unknown")
    #                .otherwise(col("LANGUAGE_CODE")))
    # )
    
    # # Trim whitespace
    # df = (
    #     df
    #     .withColumn("LANGUAGE_NAME", trim(col("LANGUAGE_NAME")))
    #     .withColumn("LANGUAGE_CODE", trim(col("LANGUAGE_CODE")))
    # )
    
    # Add silver processing timestamp
    df = df.withColumn(
        "silver_processing_timestamp", 
        current_timestamp()
    )
    
    # Select final columns
    return df.select(
        "LANGUAGE_NAME",
        "LANGUAGE_CODE",
        "ingestion_timestamp",
        "silver_processing_timestamp",
        "source_file",
        "ingestion_date"
    )

In [0]:
# @dlt.table(
#     name="silver.silver_title_ratings",
#     comment="Silver layer - Clean IMDb ratings with Decimal(3,1) precision and rating category."
# )
# @dlt.expect_or_drop("tconst_not_null", "tconst IS NOT NULL")
# def title_ratings_silver():
#     df = (
#         dlt.read_stream("bronze_title_ratings_raw")
       
#         # Remove rescued data if exists
#         .drop("_rescued_data")
       
#         # String type for identifiers
#         .withColumn("tconst", col("tconst").cast(StringType()))
       
#         # Using decimal(3,1) for averageRating
#         .withColumn("averageRating",
#             round(col("averageRating").cast("double"), 1).cast("double")
#         )
       
#         # Derived column: Rating Category based on averageRating
#         .withColumn("rating_category",
#             when(col("averageRating") <= 2.0, "Poor")
#             .when(col("averageRating") <= 4.0, "Below Average")
#             .when(col("averageRating") <= 6.0, "Average")
#             .when(col("averageRating") <= 8.0, "Good")
#             .when(col("averageRating") <= 10.0, "Excellent")
#             .otherwise("Unknown")
#         )
       
#         # Casting for integer votes
#         .withColumn("numVotes", col("numVotes").cast(IntegerType()))
       
#         # Audit column
#         .withColumn("silver_load_dt", current_timestamp())
#     )
#         # Rename all columns to UPPERCASE    
#     df = df.toDF(*[c.upper() for c in df.columns])
   
#     return df



@dlt.table(
    name="silver.silver_title_ratings",
    comment="Silver layer - Clean IMDb ratings with Decimal(3,1) precision and rating category."
)
@dlt.expect_or_drop("tconst_not_null", "tconst IS NOT NULL")
def silver_title_ratings():
    df = dlt.read_stream("bronze_title_ratings_raw")
    
    # String type for identifiers
    df = df.withColumn("tconst", col("tconst").cast(StringType()))
    
    # Using decimal(3,1) for averageRating
    df = df.withColumn("average_rating",
        round(col("averageRating").cast("double"), 1).cast("double")
    )
    
    # Derived column: Rating Category based on averageRating
    df = df.withColumn("rating_category",
        when(col("average_rating") <= 2.0, "Poor")
        .when(col("average_rating") <= 4.0, "Below Average")
        .when(col("average_rating") <= 6.0, "Average")
        .when(col("average_rating") <= 8.0, "Good")
        .when(col("average_rating") <= 10.0, "Excellent")
        .otherwise("Unknown")
    )
    
    # Casting for integer votes with renamed column
    df = df.withColumn("num_votes", col("numVotes").cast(IntegerType()))
    
    # Add audit column
    df = df.withColumn("silver_processing_timestamp", current_timestamp())
    
    # Select only required columns (drops _rescued_data, averageRating, numVotes)
    df = df.select(
        "tconst",
        "average_rating",
        "rating_category",
        "num_votes",
        "ingestion_timestamp",
        "silver_processing_timestamp",
        "source_file",
        "ingestion_date"
    )
    
    # Rename all columns to UPPERCASE    
    df = df.toDF(*[c.upper() for c in df.columns])
    
    return df

In [0]:
@dlt.table(
    name="silver.silver_title_basics",
    comment="Cleaned title basics with genres as array - IS_ADULT: 1=Adult, 0=Not Adult, -1=Unknown",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_all({
    "valid_year_range": "START_YEAR <= END_YEAR",
})
@dlt.expect_or_drop("tconst_not_null", "TCONST IS NOT NULL")
@dlt.expect_or_drop("valid_tconst", "TCONST RLIKE '^tt[0-9]{7,8}$'")
def silver_title_basics():
    """Silver transformation for title.basics with UPPERCASE column names
    
    IS_ADULT encoding:
    - 1: Adult content
    - 0: Not adult content  
    - -1: Unknown/Invalid data
    """
    
    df = (
        dlt.read_stream("bronze_title_basics_raw")  
        .drop("_rescued_data")
        
        # --- String columns: Cast and trim ---
        .withColumn("TCONST", col("tconst").cast("string"))
        .withColumn("TITLE_TYPE", col("titleType").cast("string"))
        .withColumn("PRIMARY_TITLE", trim(col("primaryTitle")).cast("string"))
        .withColumn("ORIGINAL_TITLE", trim(col("originalTitle")).cast("string"))
        
        # --- isAdult: Convert to Integer (1=adult, 0=not adult, -1=unknown) ---
        .withColumn("IS_ADULT",
            when(col("isAdult") == "1", 1)
            .when(col("isAdult") == "0", 0)
            .otherwise(-1)  # Invalid values like '2019', '\\N', or anything else
            .cast("int")
        )
        
        # --- startYear: Replace NULL with 0, cast to Integer ---
        .withColumn("START_YEAR",
            when(col("startYear").isNull(), 0)
            .otherwise(col("startYear").cast("int"))
        )
        
        # --- endYear: Replace NULL with 9999, cast to Integer ---
        .withColumn("END_YEAR",
            when(col("endYear").isNull(), 9999)
            .otherwise(col("endYear").cast("int"))
        )
        
        # --- runtimeMinutes: Replace NULL with 0, cast to Integer ---
        .withColumn("RUNTIME_MINUTES",
            when(col("runtimeMinutes").isNull(), 0)
            .otherwise(col("runtimeMinutes").cast("int"))
        )
        
        # --- genres: Replace NULL with 'unknown' ---
        .withColumn("GENRES",
            when(col("genres").isNull(), "unknown")
            .otherwise(col("genres"))
        )
        
        # # --- Create genre array ---
        # .withColumn("GENRE_ARRAY", split(col("GENRES"), ","))
        
        # --- Add silver metadata ---
        .withColumn("SILVER_PROCESSING_TIMESTAMP", current_timestamp())
    )
    
    return df.select(
        "TCONST",
        "TITLE_TYPE",
        "PRIMARY_TITLE",
        "ORIGINAL_TITLE",
        "IS_ADULT",
        "START_YEAR",
        "END_YEAR",
        "RUNTIME_MINUTES",
        "GENRES",
        "INGESTION_TIMESTAMP",
        "SILVER_PROCESSING_TIMESTAMP",
        "SOURCE_FILE",
        "INGESTION_DATE"
    )

In [0]:
import dlt
from pyspark.sql.functions import *
 
@dlt.table(
    name="silver.silver_title_crew",
    comment="Cleaned and exploded title crew data - one row per crew member (excluding Unknown)",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_all_or_drop({
    "valid_tconst": "TCONST IS NOT NULL AND TCONST RLIKE '^tt[0-9]{7,8}$'",
    "valid_crew_member": "NCONST != 'Unknown'"
})
@dlt.expect_all_or_fail({
    "tconst_not_null": "TCONST IS NOT NULL",
    "tconst_not_empty": "LENGTH(TCONST) >= 9"
})
def silver_title_crew():
    """Silver transformation for title.crew with exploded crew members - UPPERCASE columns"""
    
    df = dlt.read_stream("bronze_title_crew_raw")
    
    # STEP 1: Replace \N with NULL
    df = df.select(
        col("tconst").alias("TCONST"),
        when(col("directors") == "\\N", None).otherwise(col("directors")).alias("DIRECTORS"),
        when(col("writers") == "\\N", None).otherwise(col("writers")).alias("WRITERS"),
        col("ingestion_timestamp").alias("INGESTION_TIMESTAMP"),
        col("source_file").alias("SOURCE_FILE"),
        col("ingestion_date").alias("INGESTION_DATE")
    )
    
    # STEP 2: TRIM Whitespace
    df = (
        df
        .withColumn("DIRECTORS",
                   when(col("DIRECTORS").isNotNull(), trim(col("DIRECTORS")))
                   .otherwise(None))
        .withColumn("WRITERS",
                   when(col("WRITERS").isNotNull(), trim(col("WRITERS")))
                   .otherwise(None))
    )
    
    # STEP 3: FILTER OUT rows where BOTH directors AND writers are NULL
    df = df.filter(col("DIRECTORS").isNotNull() | col("WRITERS").isNotNull())
    
    # STEP 4: Create arrays from comma-separated strings (NO "Unknown" fallback)
    df = (
        df
        .withColumn("DIRECTOR_ARRAY",
                   when(col("DIRECTORS").isNotNull(), split(col("DIRECTORS"), ","))
                   .otherwise(array()))
        .withColumn("WRITER_ARRAY",
                   when(col("WRITERS").isNotNull(), split(col("WRITERS"), ","))
                   .otherwise(array()))
    )
    
    # STEP 5: Explode directors into separate rows (only if array is not empty)
    df_directors = (
        df
        .filter(size(col("DIRECTOR_ARRAY")) > 0)
        .select(
            "TCONST",
            explode("DIRECTOR_ARRAY").alias("NCONST"),
            "INGESTION_TIMESTAMP",
            "SOURCE_FILE",
            "INGESTION_DATE"
        )
        .withColumn("NCONST", trim(col("NCONST")))
        .withColumn("CREW_ROLE", lit("director"))
    )
    
    # STEP 6: Explode writers into separate rows (only if array is not empty)
    df_writers = (
        df
        .filter(size(col("WRITER_ARRAY")) > 0)
        .select(
            "TCONST",
            explode("WRITER_ARRAY").alias("NCONST"),
            "INGESTION_TIMESTAMP",
            "SOURCE_FILE",
            "INGESTION_DATE"
        )
        .withColumn("NCONST", trim(col("NCONST")))
        .withColumn("CREW_ROLE", lit("writer"))
    )
    
    # STEP 7: Union directors and writers
    df_exploded = df_directors.union(df_writers)
    
    # STEP 8: Add data quality flags (commented out)
    # df_exploded = (
    #     df_exploded
    #     .withColumn("IS_UNKNOWN_CREW", col("NCONST") == "Unknown")
    #     .withColumn("HAS_VALID_CREW_ID",
    #                col("NCONST").rlike("^nm[0-9]{7,8}$"))
    #     .withColumn("DATA_QUALITY_TIER",
    #                when(col("IS_UNKNOWN_CREW"), "Unknown_Crew")
    #                .when(~col("HAS_VALID_CREW_ID"), "Invalid_Crew_ID")
    #                .otherwise("Complete"))
    # )
    
    # STEP 9: ADD SILVER METADATA
    df_exploded = df_exploded.withColumn("SILVER_PROCESSING_TIMESTAMP", current_timestamp())
    
    return df_exploded

In [0]:
# @dlt.table(
#     name="silver.silver_title_principals",
#     comment="Silver layer - Cleaned principals data. Nulls handled, data types standardized, whitespace trimmed.",
#     table_properties={
#         "quality": "silver",
#         "pipelines.autoOptimize.managed": "true"
#     }
# )
# @dlt.expect_all_or_drop({
#     "tconst_not_null": "tconst IS NOT NULL",
#     "nconst_not_null": "nconst IS NOT NULL"
# })
# def silver_title_principals():
#     df = dlt.read_stream("bronze_title_principals_raw")
    
#     # Remove rescued data if exists
#     if "_rescued_data" in df.columns:
#         df = df.drop("_rescued_data")
    
#     # Handle NULL values for string columns
#     df = (
#         df
#         .withColumn("tconst", 
#                    when(col("tconst").isNull(), "unknown")
#                    .otherwise(col("tconst")))
#         .withColumn("nconst", 
#                    when(col("nconst").isNull(), "unknown")
#                    .otherwise(col("nconst")))
#         .withColumn("category", 
#                    when(col("category").isNull(), "unknown")
#                    .otherwise(col("category")))
#         .withColumn("job", 
#                    when(col("job").isNull(), "unknown")
#                    .otherwise(col("job")))
#         .withColumn("characters", 
#                    when(col("characters").isNull(), "unknown")
#                    .otherwise(col("characters")))
#     )
    
#     # Handle NULL values for numeric columns
#     df = df.withColumn("ordering", 
#                       when(col("ordering").isNull(), -1)
#                       .otherwise(col("ordering")))
    
#     # Cast to appropriate types
#     df = (
#         df
#         .withColumn("tconst", col("tconst").cast(StringType()))
#         .withColumn("ordering", col("ordering").cast(IntegerType()))
#         .withColumn("nconst", col("nconst").cast(StringType()))
#         .withColumn("category", col("category").cast(StringType()))
#         .withColumn("job", col("job").cast(StringType()))
#         .withColumn("characters", col("characters").cast(StringType()))
#     )
    
#     # Trim whitespace
#     df = (
#         df
#         .withColumn("job", trim(col("job")))
#         .withColumn("characters", trim(col("characters")))
#     )
    
#     # Add silver processing timestamp
#     df = df.withColumn(
#         "silver_load_dt", 
#         current_timestamp()
#     )
    
#         # Rename all columns to UPPERCASE    
#     df = df.toDF(*[c.upper() for c in df.columns])
    
#     return df



# =============================================================================
# SILVER LAYER - title_principals
# =============================================================================

@dlt.table(
    name="silver.silver_title_principals",
    comment="Silver layer - Cleaned principals data. Nulls handled, data types standardized, whitespace trimmed.",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_all_or_drop({
    "tconst_not_null": "tconst IS NOT NULL",
    "nconst_not_null": "nconst IS NOT NULL"
})
def silver_title_principals():
    df = dlt.read_stream("bronze_title_principals_raw")
    
    # Handle NULL values for string columns
    df = (
        df
        .withColumn("tconst", 
                   when(col("tconst").isNull(), "unknown")
                   .otherwise(col("tconst")))
        .withColumn("nconst", 
                   when(col("nconst").isNull(), "unknown")
                   .otherwise(col("nconst")))
        .withColumn("category", 
                   when(col("category").isNull(), "unknown")
                   .otherwise(col("category")))
        .withColumn("job", 
                   when(col("job").isNull(), "unknown")
                   .otherwise(col("job")))
        .withColumn("characters", 
                   when(col("characters").isNull(), "unknown")
                   .otherwise(col("characters")))
    )
    
    # Handle NULL values for numeric columns
    df = df.withColumn("ordering", 
                      when(col("ordering").isNull(), -1)
                      .otherwise(col("ordering")))
    
    # Cast to appropriate types
    df = (
        df
        .withColumn("tconst", col("tconst").cast(StringType()))
        .withColumn("ordering", col("ordering").cast(IntegerType()))
        .withColumn("nconst", col("nconst").cast(StringType()))
        .withColumn("category", col("category").cast(StringType()))
        .withColumn("job", col("job").cast(StringType()))
        .withColumn("characters", col("characters").cast(StringType()))
    )
    
    # Trim whitespace
    df = (
        df
        .withColumn("job", trim(col("job")))
        .withColumn("characters", trim(col("characters")))
    )
    
    # Add silver processing timestamp
    df = df.withColumn("silver_processing_timestamp", current_timestamp())
    
    # Select only required columns (drops _rescued_data and any extra columns)
    df = df.select(
        "tconst",
        "ordering",
        "nconst",
        "category",
        "job",
        "characters",
        "ingestion_timestamp",
        "silver_processing_timestamp",
        "source_file",
        "ingestion_date"
    )
    
    # Rename all columns to UPPERCASE    
    df = df.toDF(*[c.upper() for c in df.columns])
    
    return df



In [0]:
import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

@dlt.table(
    name="silver.silver_title_episode",
    comment="Cleaned IMDB title episode data with NULL handling and type conversions",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
@dlt.expect_all_or_drop({
    "valid_tconst": "tconst IS NOT NULL AND tconst RLIKE '^tt[0-9]{7,}$'",
    "valid_parent_tconst": "parentTconst IS NOT NULL AND parentTconst RLIKE '^tt[0-9]{7,}$'"
})
def silver_title_episode():
    """
    Cleans the bronze title_episode data.
    
    Transformations:
    - Replace \\N with NULLs
    - Replace NULL seasonNumber with -1
    - Replace NULL episodeNumber with -1
    - Cast to appropriate types
    - Validate tconst and parentTconst formats
    """
    
    df = dlt.read_stream("bronze_title_episode_raw")
    
    # Drop rescued data if exists
    if "_rescued_data" in df.columns:
        df = df.drop("_rescued_data")
    
    # Replace \N with NULLs
    df = df.select(
        *[when(col(c) == "\\N", None).otherwise(col(c)).alias(c) for c in df.columns]
    )
    
    # Handle NULL values
    df = (
        df
        .withColumn("seasonNumber", 
                   when(col("seasonNumber").isNull(), "-1")
                   .otherwise(col("seasonNumber")))
        .withColumn("episodeNumber", 
                   when(col("episodeNumber").isNull(), "-1")
                   .otherwise(col("episodeNumber")))
    )
    
    # Cast to appropriate types
    df = (
        df
        .withColumn("tconst", col("tconst").cast("string"))
        .withColumn("parentTconst", col("parentTconst").cast("string"))
        .withColumn("season_number", col("seasonNumber").cast("int"))
        .withColumn("episode_number", col("episodeNumber").cast("int"))
    )
    
    # Add silver processing timestamp
    df = df.withColumn(
        "silver_processing_timestamp", 
        current_timestamp()
    )

    # Select final columns
    df= df.select(
        "tconst",
        "parentTconst",
        "season_number",
        "episode_number",
        "ingestion_timestamp",
        "silver_processing_timestamp",
        "source_file",
        "ingestion_date"
    )
    # Rename all columns to UPPERCASE    
    df = df.toDF(*[c.upper() for c in df.columns])

    return df



In [0]:
@dlt.table(
    name="silver.silver_title_region",
    comment="IMDB title.region lookup data - pass-through from bronze",
    table_properties={
        "quality": "silver",
        "pipelines.autoOptimize.managed": "true"
    }
)
def silver_title_region():
    """
    Pass-through table from bronze to silver.
    
    Transformations:
    - No data transformations applied
    - Add silver processing timestamp for lineage tracking
    """
    
    df = dlt.read_stream("bronze_title_region_raw")
    
    # Add silver processing timestamp
    df = df.withColumn(
        "silver_processing_timestamp", 
        current_timestamp()
    )
    
    # Select final columns
    return df.select(
        "REGION_CODE",
        "REGION_NAME",
        "ingestion_timestamp",
        "silver_processing_timestamp",
        "source_file",
        "ingestion_date"
    )


In [0]:
# # =========================================================================
# # IMDB DLT Pipeline - Silver to Gold Layer (Databricks)
# # CLEAN VERSION with Table Prefixes for Ambiguous Columns
# # =========================================================================

# import dlt
# from pyspark.sql.functions import *
# from pyspark.sql.types import *

# # =========================================================================
# # TABLE PREFIX CONVENTION
# # =========================================================================
# """
# To avoid ambiguous column references, we prefix columns with table abbreviations:

# Silver Tables:
# - SP_ = silver_principals (silver_title_principals)
# - SR_ = silver_ratings (silver_title_ratings)
# - SB_ = silver_basics (silver_title_basics)
# - SE_ = silver_episode (silver_title_episode)
# - SN_ = silver_names (silver_name_basics)

# Dimension Tables:
# - DT_ = dim_title (dim_title_basics)
# - DP_ = dim_person
# - DJ_ = dim_job
# - DG_ = dim_genre
# - DR_ = dim_region
# - DL_ = dim_language
# """

# # =========================================================================
# # HELPER FUNCTION: Generate SK with Prefix using MD5 Hash
# # =========================================================================

# def generate_sk_with_prefix(df, prefix, hash_columns):
#     """
#     Generate surrogate key with prefix using MD5 hash (streaming-compatible)
    
#     Args:
#         df: Input DataFrame
#         prefix: 2-letter prefix (TB, PR, GN, etc.)
#         hash_columns: List of columns to hash for uniqueness
    
#     Returns:
#         DataFrame with SK column added
#     """
#     # Concatenate hash columns
#     hash_expr = concat_ws("||", *[coalesce(col(c).cast("string"), lit("NULL")) for c in hash_columns])
    
#     return df.withColumn(
#         "SK_HASH",
#         upper(substring(md5(hash_expr), 1, 7))
#     ).withColumn(
#         "SK",
#         concat(lit(prefix), col("SK_HASH"))
#     ).drop("SK_HASH")


# # =========================================================================
# # DIMENSION 1: DIM_TITLE_BASICS
# # =========================================================================

# @dlt.table(
#     name="dim_title_basics",
#     comment="Title basics dimension - Core title information",
#     table_properties={"quality": "gold", "type": "dimension"}
# )
# def dim_title_basics():
#     """SK based on TCONST hash"""
#     silver_df = dlt.read_stream("imdb_final_project.silver.silver_title_basics")
    
#     title_df = (
#         silver_df
#         .select("TCONST", "TITLE_TYPE", "PRIMARY_TITLE", "ORIGINAL_TITLE", 
#                 "IS_ADULT", "START_YEAR", "END_YEAR")
#         .filter(col("TCONST").isNotNull())
#         .dropDuplicates(["TCONST"])
#     )
    
#     title_df = generate_sk_with_prefix(title_df, "TB", ["TCONST"])
    
#     return title_df.select(
#         col("SK").alias("TITLE_BASICS_SK"),
#         col("TCONST").alias("TITLE_BASICS_TCONST"),
#         col("TITLE_TYPE"),
#         col("PRIMARY_TITLE"),
#         col("ORIGINAL_TITLE"),
#         col("IS_ADULT").cast("int"),
#         col("START_YEAR").cast("int"),
#         col("END_YEAR").cast("int"),
#         lit("PARAM").alias("DI_JOB_ID"),
#         current_date().alias("DI_LOAD_DATE")
#     )


# # =========================================================================
# # DIMENSION 2: DIM_GENRE
# # =========================================================================

# @dlt.table(
#     name="dim_genre",
#     comment="Genre dimension - Exploded from title basics",
#     table_properties={"quality": "gold", "type": "dimension"}
# )
# def dim_genre():
#     """SK based on GENRE_NAME hash"""
#     silver_df = dlt.read_stream("imdb_final_project.silver.silver_title_basics")
    
#     genre_df = (
#         silver_df
#         .select(explode(split(col("GENRES"), ",")).alias("GENRE_NAME"))
#         .withColumn("GENRE_NAME", trim(col("GENRE_NAME")))
#         .filter(col("GENRE_NAME").isNotNull() & (col("GENRE_NAME") != "") & (col("GENRE_NAME") != "unknown"))
#         .dropDuplicates(["GENRE_NAME"])
#     )
    
#     genre_df = generate_sk_with_prefix(genre_df, "GN", ["GENRE_NAME"])
    
#     return genre_df.select(
#         col("SK").alias("GENRE_SK"),
#         col("GENRE_NAME"),
#         lit("PARAM").alias("DI_JOB_ID"),
#         current_date().alias("DI_LOAD_DT")
#     )


# # =========================================================================
# # DIMENSION 3: DIM_PERSON
# # =========================================================================

# @dlt.table(
#     name="dim_person",
#     comment="Person dimension - Cast and crew members",
#     table_properties={"quality": "gold", "type": "dimension"}
# )
# def dim_person():
#     """SK based on NCONST hash"""
#     silver_df = dlt.read_stream("imdb_final_project.silver.silver_name_basics")
    
#     person_df = (
#         silver_df
#         .select("NCONST", "PRIMARY_NAME", "BIRTH_YEAR", "DEATH_YEAR", "IS_ALIVE")
#         .filter(col("NCONST").isNotNull())
#         .dropDuplicates(["NCONST"])
#     )
    
#     person_df = generate_sk_with_prefix(person_df, "PR", ["NCONST"])
    
#     return person_df.select(
#         col("SK").alias("PERSON_SK"),
#         col("NCONST"),
#         col("PRIMARY_NAME"),
#         col("BIRTH_YEAR").cast("int"),
#         col("DEATH_YEAR").cast("int"),
#         when(col("IS_ALIVE") == True, 1).otherwise(0).cast("int").alias("IS_ALIVE"),
#         lit("NIKHIL").alias("DI_JOB_ID"),
#         current_date().alias("DI_LOAD_DATE")
#     )


# # =========================================================================
# # DIMENSION 4: DIM_PROFESSION
# # =========================================================================

# @dlt.table(
#     name="dim_profession",
#     comment="Profession dimension - Exploded from name basics",
#     table_properties={"quality": "gold", "type": "dimension"}
# )
# def dim_profession():
#     """SK based on PROFESSION_NAME hash"""
#     silver_df = dlt.read_stream("imdb_final_project.silver.silver_name_basics")
    
#     profession_df = (
#         silver_df
#         .select(explode(split(col("PRIMARY_PROFESSION"), ",")).alias("PROFESSION_NAME"))
#         .withColumn("PROFESSION_NAME", trim(col("PROFESSION_NAME")))
#         .filter(col("PROFESSION_NAME").isNotNull() & (col("PROFESSION_NAME") != "") & (col("PROFESSION_NAME") != "Unknown"))
#         .dropDuplicates(["PROFESSION_NAME"])
#     )
    
#     profession_df = generate_sk_with_prefix(profession_df, "PF", ["PROFESSION_NAME"])
    
#     return profession_df.select(
#         col("SK").alias("PROFESSION_SK"),
#         col("PROFESSION_NAME"),
#         lit("NIKHIL").alias("DI_JOB_ID"),
#         current_date().alias("DI_LOAD_DATE")
#     )


# # =========================================================================
# # DIMENSION 5: DIM_JOB
# # =========================================================================

# @dlt.table(
#     name="dim_job",
#     comment="Job category dimension from title principals",
#     table_properties={"quality": "gold", "type": "dimension"}
# )
# def dim_job():
#     """SK based on JOB_CATEGORY hash"""
#     silver_df = dlt.read_stream("imdb_final_project.silver.silver_title_principals")
    
#     job_df = (
#         silver_df
#         .select(col("CATEGORY").alias("JOB_CATEGORY"))
#         .filter(col("JOB_CATEGORY").isNotNull() & (col("JOB_CATEGORY") != "unknown"))
#         .dropDuplicates(["JOB_CATEGORY"])
#     )
    
#     job_df = generate_sk_with_prefix(job_df, "JB", ["JOB_CATEGORY"])
    
#     return job_df.select(
#         col("SK").alias("JOB_SK"),
#         col("JOB_CATEGORY"),
#         lit("PRATHUSH").alias("DI_JOB_ID"),
#         current_date().alias("DI_LOAD_DATE")
#     )


# # =========================================================================
# # DIMENSION 6: DIM_REGION
# # =========================================================================

# @dlt.table(
#     name="dim_region",
#     comment="Region dimension from title region data",
#     table_properties={"quality": "gold", "type": "dimension"}
# )
# def dim_region():
#     """SK based on REGION_CODE hash"""
#     silver_df = dlt.read_stream("imdb_final_project.silver.silver_title_region")
    
#     region_df = (
#         silver_df
#         .select("REGION_CODE", "REGION_NAME")
#         .filter(col("REGION_CODE").isNotNull())
#         .dropDuplicates(["REGION_CODE"])
#     )
    
#     region_df = generate_sk_with_prefix(region_df, "RG", ["REGION_CODE"])
    
#     return region_df.select(
#         col("SK").alias("REGION_SK"),
#         col("REGION_CODE"),
#         col("REGION_NAME"),
#         lit("PRATHUSH").alias("DI_JOB_ID"),
#         current_date().alias("DI_LOAD_DT")
#     )


# # =========================================================================
# # DIMENSION 7: DIM_LANGUAGE
# # =========================================================================

# @dlt.table(
#     name="dim_language",
#     comment="Language dimension from language codes data",
#     table_properties={"quality": "gold", "type": "dimension"}
# )
# def dim_language():
#     """SK based on LANGUAGE_CODE hash"""
#     silver_df = dlt.read_stream("imdb_final_project.silver.silver_title_language_codes")
    
#     language_df = (
#         silver_df
#         .select("LANGUAGE_CODE", "LANGUAGE_NAME")
#         .filter(col("LANGUAGE_CODE").isNotNull())
#         .dropDuplicates(["LANGUAGE_CODE"])
#     )
    
#     language_df = generate_sk_with_prefix(language_df, "LG", ["LANGUAGE_CODE"])
    
#     return language_df.select(
#         col("SK").alias("LANGUAGE_SK"),
#         col("LANGUAGE_CODE"),
#         col("LANGUAGE_NAME"),
#         lit("PRATHUSH").alias("DI_JOB_ID"),
#         current_date().alias("DI_LOAD_DATE")
#     )


# # =========================================================================
# # DIMENSION 8: DIM_TITLE_AKAS
# # =========================================================================

# @dlt.table(
#     name="dim_title_akas",
#     comment="Title AKAS dimension - Title translations with FK lookups",
#     table_properties={"quality": "gold", "type": "dimension"}
# )
# def dim_title_akas():
#     """SK based on TITLE_ID + ORDERING hash"""
#     # Read from silver
#     silver_akas = dlt.read_stream("imdb_final_project.silver.silver_title_akas")
    
#     # Add SA_ prefix to silver_akas columns
#     sa_df = silver_akas.select(
#         col("TITLE_ID").alias("SA_TITLE_ID"),
#         col("ORDERING").alias("SA_ORDERING"),
#         col("TITLE").alias("SA_TITLE"),
#         col("REGION").alias("SA_REGION"),
#         col("LANGUAGE").alias("SA_LANGUAGE"),
#         col("TYPES").alias("SA_TYPES"),
#         col("ATTRIBUTES").alias("SA_ATTRIBUTES"),
#         col("IS_ORIGINAL_TITLE").alias("SA_IS_ORIGINAL_TITLE")
#     )
    
#     # Read dimensions with prefixes
#     dt_df = dlt.read("dim_title_basics").select(
#         col("TITLE_BASICS_SK"),
#         col("TITLE_BASICS_TCONST").alias("DT_TCONST")
#     )
    
#     dr_df = dlt.read("dim_region").select(
#         col("REGION_SK"),
#         col("REGION_CODE").alias("DR_REGION_CODE")
#     )
    
#     dl_df = dlt.read("dim_language").select(
#         col("LANGUAGE_SK"),
#         col("LANGUAGE_CODE").alias("DL_LANGUAGE_CODE")
#     )
    
#     # Join with prefixed columns
#     akas_df = (
#         sa_df
#         .join(dt_df, col("SA_TITLE_ID") == col("DT_TCONST"), "left")
#         .join(dr_df, col("SA_REGION") == col("DR_REGION_CODE"), "left")
#         .join(dl_df, col("SA_LANGUAGE") == col("DL_LANGUAGE_CODE"), "left")
#     )
    
#     # Generate SK using SA_ prefixed columns
#     akas_df = generate_sk_with_prefix(akas_df, "TA", ["SA_TITLE_ID", "SA_ORDERING"])
    
#     return akas_df.select(
#         col("SK").alias("TITLE_AKA_SK"),
#         col("TITLE_BASICS_SK"),
#         col("SA_TITLE").alias("AKA_TITLE"),
#         col("SA_TYPES").alias("TITLE_AKAS_TYPES"),
#         col("SA_ATTRIBUTES").alias("ATTRIBUTES"),
#         col("SA_IS_ORIGINAL_TITLE").cast("int").alias("IS_ORIGINAL_TITLE"),
#         col("REGION_SK"),
#         col("LANGUAGE_SK"),
#         lit("NIKHIL").alias("DI_JOB_ID"),
#         current_date().alias("DI_LOAD_DATE")
#     )


# # =========================================================================
# # BRIDGE 1: BRIDGE_TITLE_GENRE
# # =========================================================================

# @dlt.table(
#     name="bridge_title_genre",
#     comment="Bridge table linking titles to genres",
#     table_properties={"quality": "gold", "type": "bridge"}
# )
# def bridge_title_genre():
#     """Bridge between DIM_TITLE_BASICS and DIM_GENRE"""
#     silver_basics = dlt.read_stream("imdb_final_project.silver.silver_title_basics")
    
#     # Add SB_ prefix
#     sb_df = silver_basics.select(
#         col("TCONST").alias("SB_TCONST"),
#         col("GENRES").alias("SB_GENRES")
#     )
    
#     # Explode genres
#     exploded_df = (
#         sb_df
#         .select(
#             col("SB_TCONST"),
#             explode(split(col("SB_GENRES"), ",")).alias("SB_GENRE_NAME")
#         )
#         .withColumn("SB_GENRE_NAME", trim(col("SB_GENRE_NAME")))
#         .filter(col("SB_GENRE_NAME").isNotNull() & (col("SB_GENRE_NAME") != "") & (col("SB_GENRE_NAME") != "unknown"))
#     )
    
#     # Read dimensions with prefixes
#     dt_df = dlt.read("dim_title_basics").select(
#         col("TITLE_BASICS_SK"),
#         col("TITLE_BASICS_TCONST").alias("DT_TCONST")
#     )
    
#     dg_df = dlt.read("dim_genre").select(
#         col("GENRE_SK"),
#         col("GENRE_NAME").alias("DG_GENRE_NAME")
#     )
    
#     # Join
#     bridge_df = (
#         exploded_df
#         .join(dt_df, col("SB_TCONST") == col("DT_TCONST"), "inner")
#         .join(dg_df, col("SB_GENRE_NAME") == col("DG_GENRE_NAME"), "inner")
#         .select("TITLE_BASICS_SK", "GENRE_SK")
#         .dropDuplicates(["TITLE_BASICS_SK", "GENRE_SK"])
#     )
    
#     return bridge_df.select(
#         col("TITLE_BASICS_SK"),
#         col("GENRE_SK"),
#         lit("PARAM").alias("DI_JOB_ID"),
#         current_date().alias("DI_LOAD_DATE")
#     )


# # =========================================================================
# # BRIDGE 2: BRIDGE_PERSON_PROFESSION
# # =========================================================================

# @dlt.table(
#     name="bridge_person_profession",
#     comment="Bridge table linking persons to professions",
#     table_properties={"quality": "gold", "type": "bridge"}
# )
# def bridge_person_profession():
#     """Bridge between DIM_PERSON and DIM_PROFESSION"""
#     silver_names = dlt.read_stream("imdb_final_project.silver.silver_name_basics")
    
#     # Add SN_ prefix
#     sn_df = silver_names.select(
#         col("NCONST").alias("SN_NCONST"),
#         col("PRIMARY_PROFESSION").alias("SN_PRIMARY_PROFESSION")
#     )
    
#     # Explode professions
#     exploded_df = (
#         sn_df
#         .select(
#             col("SN_NCONST"),
#             explode(split(col("SN_PRIMARY_PROFESSION"), ",")).alias("SN_PROFESSION_NAME")
#         )
#         .withColumn("SN_PROFESSION_NAME", trim(col("SN_PROFESSION_NAME")))
#         .filter(col("SN_PROFESSION_NAME").isNotNull() & (col("SN_PROFESSION_NAME") != "") & (col("SN_PROFESSION_NAME") != "Unknown"))
#     )
    
#     # Read dimensions with prefixes
#     dp_df = dlt.read("dim_person").select(
#         col("PERSON_SK"),
#         col("NCONST").alias("DP_NCONST")
#     )
    
#     dpf_df = dlt.read("dim_profession").select(
#         col("PROFESSION_SK"),
#         col("PROFESSION_NAME").alias("DPF_PROFESSION_NAME")
#     )
    
#     # Join
#     bridge_df = (
#         exploded_df
#         .join(dp_df, col("SN_NCONST") == col("DP_NCONST"), "inner")
#         .join(dpf_df, col("SN_PROFESSION_NAME") == col("DPF_PROFESSION_NAME"), "inner")
#         .select("PERSON_SK", "PROFESSION_SK")
#         .dropDuplicates(["PERSON_SK", "PROFESSION_SK"])
#     )
    
#     return bridge_df.select(
#         col("PERSON_SK"),
#         col("PROFESSION_SK"),
#         lit("NIKHIL").alias("DI_JOB_ID"),
#         current_date().alias("DI_LOAD_DATE")
#     )


# # =========================================================================
# # FACT 1: FACT_TITLE_PARTICIPATION
# # =========================================================================

# @dlt.table(
#     name="fact_title_participation",
#     comment="Fact table for title participation - person-title-job grain",
#     table_properties={"quality": "gold", "type": "fact"}
# )
# def fact_title_participation():
#     """SK based on TCONST + NCONST + ORDERING hash"""
#     silver_principals = dlt.read_stream("imdb_final_project.silver.silver_title_principals")
    
#     # Add SP_ prefix to silver_principals columns
#     sp_df = silver_principals.select(
#         col("TCONST").alias("SP_TCONST"),
#         col("NCONST").alias("SP_NCONST"),
#         col("CATEGORY").alias("SP_CATEGORY"),
#         col("ORDERING").alias("SP_ORDERING"),
#         col("CHARACTERS").alias("SP_CHARACTERS")
#     )
    
#     # Read dimensions with prefixes
#     dt_df = dlt.read("dim_title_basics").select(
#         col("TITLE_BASICS_SK"),
#         col("TITLE_BASICS_TCONST").alias("DT_TCONST")
#     )
    
#     dp_df = dlt.read("dim_person").select(
#         col("PERSON_SK"),
#         col("NCONST").alias("DP_NCONST")
#     )
    
#     dj_df = dlt.read("dim_job").select(
#         col("JOB_SK"),
#         col("JOB_CATEGORY").alias("DJ_CATEGORY")
#     )
    
#     # Join with prefixed columns - no ambiguity!
#     fact_df = (
#         sp_df
#         .join(dt_df, col("SP_TCONST") == col("DT_TCONST"), "inner")
#         .join(dp_df, col("SP_NCONST") == col("DP_NCONST"), "inner")
#         .join(dj_df, col("SP_CATEGORY") == col("DJ_CATEGORY"), "inner")
#     )
    
#     # Generate SK using SP_ prefixed columns
#     fact_df = generate_sk_with_prefix(fact_df, "FP", ["SP_TCONST", "SP_NCONST", "SP_ORDERING"])
    
#     return fact_df.select(
#         col("SK").alias("PARTICIPATION_SK"),
#         col("JOB_SK"),
#         col("TITLE_BASICS_SK"),
#         col("PERSON_SK"),
#         col("SP_CHARACTERS").alias("CHARACTER_NAME"),
#         lit("PRATHUSH").alias("DI_JOB_ID"),
#         current_date().alias("DI_LOAD_DATE")
#     )


# # =========================================================================
# # FACT 2: FACT_TITLE_STATS
# # =========================================================================

# @dlt.table(
#     name="fact_title_stats",
#     comment="Fact table for title statistics - ratings, runtime, episodes",
#     table_properties={"quality": "gold", "type": "fact"}
# )
# def fact_title_stats():
#     """SK based on TCONST + SEASON_NUMBER hash"""
#     silver_ratings = dlt.read_stream("imdb_final_project.silver.silver_title_ratings")
#     silver_basics = dlt.read_stream("imdb_final_project.silver.silver_title_basics")
#     silver_episode = dlt.read_stream("imdb_final_project.silver.silver_title_episode")
    
#     # Add prefixes to silver tables
#     sr_df = silver_ratings.select(
#         col("TCONST").alias("SR_TCONST"),
#         col("AVERAGE_RATING").alias("SR_AVERAGE_RATING"),
#         col("RATING_CATEGORY").alias("SR_RATING_CATEGORY"),
#         col("NUM_VOTES").alias("SR_NUM_VOTES")
#     )
    
#     sb_df = silver_basics.select(
#         col("TCONST").alias("SB_TCONST"),
#         col("RUNTIME_MINUTES").alias("SB_RUNTIME_MINUTES")
#     )
    
#     # Aggregate episodes with SE_ prefix
#     episode_agg = (
#         silver_episode
#         .groupBy(
#             col("PARENTTCONST").alias("SE_PARENTTCONST"),
#             col("SEASON_NUMBER").alias("SE_SEASON_NUMBER")
#         )
#         .agg(count("TCONST").alias("SE_NO_OF_EPISODES"))
#     )
    
#     # Read dimension with prefix
#     dt_df = dlt.read("dim_title_basics").select(
#         col("TITLE_BASICS_SK"),
#         col("TITLE_BASICS_TCONST").alias("DT_TCONST")
#     )
    
#     # Join all with prefixed columns
#     fact_df = (
#         sr_df
#         .join(dt_df, col("SR_TCONST") == col("DT_TCONST"), "inner")
#         .join(sb_df, col("SR_TCONST") == col("SB_TCONST"), "left")
#         .join(episode_agg, col("SR_TCONST") == col("SE_PARENTTCONST"), "left")
#     )
    
#     # Generate SK using SR_ prefixed columns
#     fact_df = generate_sk_with_prefix(fact_df, "FS", ["SR_TCONST", "SE_SEASON_NUMBER"])
    
#     return fact_df.select(
#         col("SK").alias("TITLE_STATS_SK"),
#         col("TITLE_BASICS_SK"),
#         col("SR_AVERAGE_RATING").cast("decimal(3,1)").alias("AVERAGE_RATING"),
#         col("SR_RATING_CATEGORY").alias("RATING_CATEGORY"),
#         col("SR_NUM_VOTES").cast("int").alias("NUM_VOTES"),
#         col("SB_RUNTIME_MINUTES").cast("int").alias("RUNTIME_MINUTES"),
#         col("SE_SEASON_NUMBER").cast("int").alias("SEASON_NUMBER"),
#         col("SE_NO_OF_EPISODES").cast("int").alias("NO_OF_EPISODES"),
#         lit("PRATHUSH").alias("DI_JOB_ID"),
#         current_date().alias("DI_LOAD_DATE")
#     )

In [0]:
# =========================================================================
# IMDB DLT Pipeline - Silver to Gold Layer (Databricks)
# CLEAN VERSION with Table Prefixes for Ambiguous Columns
# =========================================================================

import dlt
from pyspark.sql.functions import *
from pyspark.sql.types import *

# =========================================================================
# TABLE PREFIX CONVENTION
# =========================================================================
"""
To avoid ambiguous column references, we prefix columns with table abbreviations:

Silver Tables:
- SP_ = silver_principals (silver_title_principals)
- SR_ = silver_ratings (silver_title_ratings)
- SB_ = silver_basics (silver_title_basics)
- SE_ = silver_episode (silver_title_episode)
- SN_ = silver_names (silver_name_basics)

Dimension Tables:
- DT_ = dim_title (dim_title_basics)
- DP_ = dim_person
- DJ_ = dim_job
- DG_ = dim_genre
- DR_ = dim_region
- DL_ = dim_language
"""

# =========================================================================
# HELPER FUNCTION: Generate SK with Prefix using MD5 Hash
# =========================================================================

def generate_sk_with_prefix(df, prefix, hash_columns):
    """
    Generate surrogate key with prefix using MD5 hash (streaming-compatible)
    
    Args:
        df: Input DataFrame
        prefix: 2-letter prefix (TB, PR, GN, etc.)
        hash_columns: List of columns to hash for uniqueness
    
    Returns:
        DataFrame with SK column added
    """
    # Concatenate hash columns
    hash_expr = concat_ws("||", *[coalesce(col(c).cast("string"), lit("NULL")) for c in hash_columns])
    
    return df.withColumn(
        "SK_HASH",
        upper(substring(md5(hash_expr), 1, 7))
    ).withColumn(
        "SK",
        concat(lit(prefix), col("SK_HASH"))
    ).drop("SK_HASH")


# =========================================================================
# DIMENSION 1: DIM_TITLE_BASICS
# =========================================================================

@dlt.table(
    name="gold.dim_title_basics",
    comment="Title basics dimension - Core title information",
    table_properties={"quality": "gold", "type": "dimension"}
)
def dim_title_basics():
    """SK based on TCONST hash"""
    silver_df = dlt.read_stream("imdb_final_project.silver.silver_title_basics")
    
    title_df = (
        silver_df
        .select("TCONST", "TITLE_TYPE", "PRIMARY_TITLE", "ORIGINAL_TITLE", 
                "IS_ADULT", "START_YEAR", "END_YEAR")
        .filter(col("TCONST").isNotNull())
        .dropDuplicates(["TCONST"])
    )
    
    title_df = generate_sk_with_prefix(title_df, "TB", ["TCONST"])
    
    return title_df.select(
        col("SK").alias("TITLE_BASICS_SK"),
        col("TCONST").alias("TITLE_BASICS_TCONST"),
        col("TITLE_TYPE"),
        col("PRIMARY_TITLE"),
        col("ORIGINAL_TITLE"),
        col("IS_ADULT").cast("int"),
        col("START_YEAR").cast("int"),
        col("END_YEAR").cast("int"),
        lit("PARAM").alias("DI_JOB_ID"),
        current_date().alias("DI_LOAD_DATE")
    )


# =========================================================================
# DIMENSION 2: DIM_GENRE
# =========================================================================

@dlt.table(
    name="gold.dim_genre",
    comment="Genre dimension - Exploded from title basics",
    table_properties={"quality": "gold", "type": "dimension"}
)
def dim_genre():
    """SK based on GENRE_NAME hash"""
    silver_df = dlt.read_stream("imdb_final_project.silver.silver_title_basics")
    
    genre_df = (
        silver_df
        .select(explode(split(col("GENRES"), ",")).alias("GENRE_NAME"))
        .withColumn("GENRE_NAME", trim(col("GENRE_NAME")))
        .filter(col("GENRE_NAME").isNotNull() & (col("GENRE_NAME") != "") & (col("GENRE_NAME") != "unknown"))
        .dropDuplicates(["GENRE_NAME"])
    )
    
    genre_df = generate_sk_with_prefix(genre_df, "GN", ["GENRE_NAME"])
    
    return genre_df.select(
        col("SK").alias("GENRE_SK"),
        col("GENRE_NAME"),
        lit("PARAM").alias("DI_JOB_ID"),
        current_date().alias("DI_LOAD_DT")
    )


# =========================================================================
# DIMENSION 3: DIM_PERSON
# =========================================================================

@dlt.table(
    name="gold.dim_person",
    comment="Person dimension - Cast and crew members",
    table_properties={"quality": "gold", "type": "dimension"}
)
def dim_person():
    """SK based on NCONST hash"""
    silver_df = dlt.read_stream("imdb_final_project.silver.silver_name_basics")
    
    person_df = (
        silver_df
        .select("NCONST", "PRIMARY_NAME", "BIRTH_YEAR", "DEATH_YEAR", "IS_ALIVE")
        .filter(col("NCONST").isNotNull())
        .dropDuplicates(["NCONST"])
    )
    
    person_df = generate_sk_with_prefix(person_df, "PR", ["NCONST"])
    
    return person_df.select(
        col("SK").alias("PERSON_SK"),
        col("NCONST"),
        col("PRIMARY_NAME"),
        col("BIRTH_YEAR").cast("int"),
        col("DEATH_YEAR").cast("int"),
        when(col("IS_ALIVE") == True, 1).otherwise(0).cast("int").alias("IS_ALIVE"),
        lit("NIKHIL").alias("DI_JOB_ID"),
        current_date().alias("DI_LOAD_DATE")
    )


# =========================================================================
# DIMENSION 4: DIM_PROFESSION
# =========================================================================

@dlt.table(
    name="gold.dim_profession",
    comment="Profession dimension - Exploded from name basics",
    table_properties={"quality": "gold", "type": "dimension"}
)
def dim_profession():
    """SK based on PROFESSION_NAME hash"""
    silver_df = dlt.read_stream("imdb_final_project.silver.silver_name_basics")
    
    profession_df = (
        silver_df
        .select(explode(split(col("PRIMARY_PROFESSION"), ",")).alias("PROFESSION_NAME"))
        .withColumn("PROFESSION_NAME", trim(col("PROFESSION_NAME")))
        .filter(col("PROFESSION_NAME").isNotNull() & (col("PROFESSION_NAME") != "") & (col("PROFESSION_NAME") != "Unknown"))
        .dropDuplicates(["PROFESSION_NAME"])
    )
    
    profession_df = generate_sk_with_prefix(profession_df, "PF", ["PROFESSION_NAME"])
    
    return profession_df.select(
        col("SK").alias("PROFESSION_SK"),
        col("PROFESSION_NAME"),
        lit("NIKHIL").alias("DI_JOB_ID"),
        current_date().alias("DI_LOAD_DATE")
    )


# =========================================================================
# DIMENSION 5: DIM_JOB
# =========================================================================

@dlt.table(
    name="gold.dim_job",
    comment="Job category dimension from title principals",
    table_properties={"quality": "gold", "type": "dimension"}
)
def dim_job():
    """SK based on JOB_CATEGORY hash"""
    silver_df = dlt.read_stream("imdb_final_project.silver.silver_title_principals")
    
    job_df = (
        silver_df
        .select(col("CATEGORY").alias("JOB_CATEGORY"))
        .filter(col("JOB_CATEGORY").isNotNull() & (col("JOB_CATEGORY") != "unknown"))
        .dropDuplicates(["JOB_CATEGORY"])
    )
    
    job_df = generate_sk_with_prefix(job_df, "JB", ["JOB_CATEGORY"])
    
    return job_df.select(
        col("SK").alias("JOB_SK"),
        col("JOB_CATEGORY"),
        lit("PRATHUSH").alias("DI_JOB_ID"),
        current_date().alias("DI_LOAD_DATE")
    )


# =========================================================================
# DIMENSION 6: DIM_REGION
# =========================================================================

@dlt.table(
    name="gold.dim_region",
    comment="Region dimension from title region data",
    table_properties={"quality": "gold", "type": "dimension"}
)
def dim_region():
    """SK based on REGION_CODE hash"""
    silver_df = dlt.read_stream("imdb_final_project.silver.silver_title_region")
    
    region_df = (
        silver_df
        .select("REGION_CODE", "REGION_NAME")
        .filter(col("REGION_CODE").isNotNull())
        .dropDuplicates(["REGION_CODE"])
    )
    
    region_df = generate_sk_with_prefix(region_df, "RG", ["REGION_CODE"])
    
    return region_df.select(
        col("SK").alias("REGION_SK"),
        col("REGION_CODE"),
        col("REGION_NAME"),
        lit("PRATHUSH").alias("DI_JOB_ID"),
        current_date().alias("DI_LOAD_DT")
    )


# =========================================================================
# DIMENSION 7: DIM_LANGUAGE
# =========================================================================

@dlt.table(
    name="gold.dim_language",
    comment="Language dimension from language codes data",
    table_properties={"quality": "gold", "type": "dimension"}
)
def dim_language():
    """SK based on LANGUAGE_CODE hash"""
    silver_df = dlt.read_stream("imdb_final_project.silver.silver_title_language_codes")
    
    language_df = (
        silver_df
        .select("LANGUAGE_CODE", "LANGUAGE_NAME")
        .filter(col("LANGUAGE_CODE").isNotNull())
        .dropDuplicates(["LANGUAGE_CODE"])
    )
    
    language_df = generate_sk_with_prefix(language_df, "LG", ["LANGUAGE_CODE"])
    
    return language_df.select(
        col("SK").alias("LANGUAGE_SK"),
        col("LANGUAGE_CODE"),
        col("LANGUAGE_NAME"),
        lit("PRATHUSH").alias("DI_JOB_ID"),
        current_date().alias("DI_LOAD_DATE")
    )


# =========================================================================
# DIMENSION 8: DIM_TITLE_AKAS
# =========================================================================

@dlt.table(
    name="gold.dim_title_akas",
    comment="Title AKAS dimension - Title translations with FK lookups",
    table_properties={"quality": "gold", "type": "dimension"}
)
def dim_title_akas():
    """SK based on TITLE_ID + ORDERING hash"""
    # Read from silver
    silver_akas = dlt.read_stream("imdb_final_project.silver.silver_title_akas")
    
    # Add SA_ prefix to silver_akas columns
    sa_df = silver_akas.select(
        col("TITLE_ID").alias("SA_TITLE_ID"),
        col("ORDERING").alias("SA_ORDERING"),
        col("TITLE").alias("SA_TITLE"),
        col("REGION").alias("SA_REGION"),
        col("LANGUAGE").alias("SA_LANGUAGE"),
        col("TYPES").alias("SA_TYPES"),
        col("ATTRIBUTES").alias("SA_ATTRIBUTES"),
        col("IS_ORIGINAL_TITLE").alias("SA_IS_ORIGINAL_TITLE")
    )
    
    # Read dimensions with prefixes
    dt_df = dlt.read("gold.dim_title_basics").select(
        col("TITLE_BASICS_SK"),
        col("TITLE_BASICS_TCONST").alias("DT_TCONST")
    )
    
    dr_df = dlt.read("gold.dim_region").select(
        col("REGION_SK"),
        col("REGION_CODE").alias("DR_REGION_CODE")
    )
    
    dl_df = dlt.read("gold.dim_language").select(
        col("LANGUAGE_SK"),
        col("LANGUAGE_CODE").alias("DL_LANGUAGE_CODE")
    )
    
    # Join with prefixed columns
    akas_df = (
        sa_df
        .join(dt_df, col("SA_TITLE_ID") == col("DT_TCONST"), "left")
        .join(dr_df, col("SA_REGION") == col("DR_REGION_CODE"), "left")
        .join(dl_df, col("SA_LANGUAGE") == col("DL_LANGUAGE_CODE"), "left")
    )
    
    # Generate SK using SA_ prefixed columns
    akas_df = generate_sk_with_prefix(akas_df, "TA", ["SA_TITLE_ID", "SA_ORDERING"])
    
    return akas_df.select(
        col("SK").alias("TITLE_AKA_SK"),
        col("TITLE_BASICS_SK"),
        col("SA_TITLE").alias("AKA_TITLE"),
        col("SA_TYPES").alias("TITLE_AKAS_TYPES"),
        col("SA_ATTRIBUTES").alias("ATTRIBUTES"),
        col("SA_IS_ORIGINAL_TITLE").cast("int").alias("IS_ORIGINAL_TITLE"),
        coalesce(col("REGION_SK"), lit("-1")).alias("REGION_SK"),
        coalesce(col("LANGUAGE_SK"), lit("-1")).alias("LANGUAGE_SK"),
        lit("NIKHIL").alias("DI_JOB_ID"),
        current_date().alias("DI_LOAD_DATE")
    )


# =========================================================================
# BRIDGE 1: BRIDGE_TITLE_GENRE
# =========================================================================

@dlt.table(
    name="gold.bridge_title_genre",
    comment="Bridge table linking titles to genres",
    table_properties={"quality": "gold", "type": "bridge"}
)
def bridge_title_genre():
    """Bridge between DIM_TITLE_BASICS and DIM_GENRE"""
    silver_basics = dlt.read_stream("imdb_final_project.silver.silver_title_basics")
    
    # Add SB_ prefix
    sb_df = silver_basics.select(
        col("TCONST").alias("SB_TCONST"),
        col("GENRES").alias("SB_GENRES")
    )
    
    # Explode genres
    exploded_df = (
        sb_df
        .select(
            col("SB_TCONST"),
            explode(split(col("SB_GENRES"), ",")).alias("SB_GENRE_NAME")
        )
        .withColumn("SB_GENRE_NAME", trim(col("SB_GENRE_NAME")))
        .filter(col("SB_GENRE_NAME").isNotNull() & (col("SB_GENRE_NAME") != "") & (col("SB_GENRE_NAME") != "unknown"))
    )
    
    # Read dimensions with prefixes
    dt_df = dlt.read("gold.dim_title_basics").select(
        col("TITLE_BASICS_SK"),
        col("TITLE_BASICS_TCONST").alias("DT_TCONST")
    )
    
    dg_df = dlt.read("gold.dim_genre").select(
        col("GENRE_SK"),
        col("GENRE_NAME").alias("DG_GENRE_NAME")
    )
    
    # Join
    bridge_df = (
        exploded_df
        .join(dt_df, col("SB_TCONST") == col("DT_TCONST"), "inner")
        .join(dg_df, col("SB_GENRE_NAME") == col("DG_GENRE_NAME"), "inner")
        .select("TITLE_BASICS_SK", "GENRE_SK")
        .dropDuplicates(["TITLE_BASICS_SK", "GENRE_SK"])
    )
    
    return bridge_df.select(
        col("TITLE_BASICS_SK"),
        col("GENRE_SK"),
        lit("PARAM").alias("DI_JOB_ID"),
        current_date().alias("DI_LOAD_DATE")
    )


# =========================================================================
# BRIDGE 2: BRIDGE_PERSON_PROFESSION
# =========================================================================

@dlt.table(
    name="gold.bridge_person_profession",
    comment="Bridge table linking persons to professions",
    table_properties={"quality": "gold", "type": "bridge"}
)
def bridge_person_profession():
    """Bridge between DIM_PERSON and DIM_PROFESSION"""
    silver_names = dlt.read_stream("imdb_final_project.silver.silver_name_basics")
    
    # Add SN_ prefix
    sn_df = silver_names.select(
        col("NCONST").alias("SN_NCONST"),
        col("PRIMARY_PROFESSION").alias("SN_PRIMARY_PROFESSION")
    )
    
    # Explode professions
    exploded_df = (
        sn_df
        .select(
            col("SN_NCONST"),
            explode(split(col("SN_PRIMARY_PROFESSION"), ",")).alias("SN_PROFESSION_NAME")
        )
        .withColumn("SN_PROFESSION_NAME", trim(col("SN_PROFESSION_NAME")))
        .filter(col("SN_PROFESSION_NAME").isNotNull() & (col("SN_PROFESSION_NAME") != "") & (col("SN_PROFESSION_NAME") != "Unknown"))
    )
    
    # Read dimensions with prefixes
    dp_df = dlt.read("gold.dim_person").select(
        col("PERSON_SK"),
        col("NCONST").alias("DP_NCONST")
    )
    
    dpf_df = dlt.read("gold.dim_profession").select(
        col("PROFESSION_SK"),
        col("PROFESSION_NAME").alias("DPF_PROFESSION_NAME")
    )
    
    # Join
    bridge_df = (
        exploded_df
        .join(dp_df, col("SN_NCONST") == col("DP_NCONST"), "inner")
        .join(dpf_df, col("SN_PROFESSION_NAME") == col("DPF_PROFESSION_NAME"), "inner")
        .select("PERSON_SK", "PROFESSION_SK")
        .dropDuplicates(["PERSON_SK", "PROFESSION_SK"])
    )
    
    return bridge_df.select(
        col("PERSON_SK"),
        col("PROFESSION_SK"),
        lit("NIKHIL").alias("DI_JOB_ID"),
        current_date().alias("DI_LOAD_DATE")
    )


# =========================================================================
# FACT 1: FACT_TITLE_PARTICIPATION
# =========================================================================

@dlt.table(
    name="gold.fact_title_participation",
    comment="Fact table for title participation - person-title-job grain",
    table_properties={"quality": "gold", "type": "fact"}
)
def fact_title_participation():
    """SK based on TCONST + NCONST + ORDERING hash"""
    silver_principals = dlt.read_stream("imdb_final_project.silver.silver_title_principals")
    
    # Add SP_ prefix to silver_principals columns
    sp_df = silver_principals.select(
        col("TCONST").alias("SP_TCONST"),
        col("NCONST").alias("SP_NCONST"),
        col("CATEGORY").alias("SP_CATEGORY"),
        col("ORDERING").alias("SP_ORDERING"),
        col("CHARACTERS").alias("SP_CHARACTERS")
    )
    
    # Read dimensions with prefixes
    dt_df = dlt.read("gold.dim_title_basics").select(
        col("TITLE_BASICS_SK"),
        col("TITLE_BASICS_TCONST").alias("DT_TCONST")
    )
    
    dp_df = dlt.read("gold.dim_person").select(
        col("PERSON_SK"),
        col("NCONST").alias("DP_NCONST")
    )
    
    dj_df = dlt.read("gold.dim_job").select(
        col("JOB_SK"),
        col("JOB_CATEGORY").alias("DJ_CATEGORY")
    )
    
    # Join with prefixed columns - no ambiguity!
    fact_df = (
        sp_df
        .join(dt_df, col("SP_TCONST") == col("DT_TCONST"), "inner")
        .join(dp_df, col("SP_NCONST") == col("DP_NCONST"), "inner")
        .join(dj_df, col("SP_CATEGORY") == col("DJ_CATEGORY"), "inner")
    )
    
    # Generate SK using SP_ prefixed columns
    fact_df = generate_sk_with_prefix(fact_df, "FP", ["SP_TCONST", "SP_NCONST", "SP_ORDERING"])
    
    return fact_df.select(
        col("SK").alias("PARTICIPATION_SK"),
        col("JOB_SK"),
        col("TITLE_BASICS_SK"),
        col("PERSON_SK"),
        col("SP_CHARACTERS").alias("CHARACTER_NAME"),
        lit("PRATHUSH").alias("DI_JOB_ID"),
        current_date().alias("DI_LOAD_DATE")
    )


# =========================================================================
# FACT 2: FACT_TITLE_STATS
# =========================================================================

@dlt.table(
    name="gold.fact_title_stats",
    comment="Fact table for title statistics - ratings, runtime, episodes",
    table_properties={"quality": "gold", "type": "fact"}
)
def fact_title_stats():
    """SK based on TCONST + SEASON_NUMBER hash"""
    # Read ratings as STREAMING (main source)
    silver_ratings = dlt.read_stream("imdb_final_project.silver.silver_title_ratings")
    
    # Read basics and episode as BATCH for aggregation (not streaming)
    silver_basics = spark.read.table("imdb_final_project.silver.silver_title_basics")
    silver_episode = spark.read.table("imdb_final_project.silver.silver_title_episode")
    
    # Add prefixes to silver tables
    sr_df = silver_ratings.select(
        col("TCONST").alias("SR_TCONST"),
        col("AVERAGE_RATING").alias("SR_AVERAGE_RATING"),
        col("RATING_CATEGORY").alias("SR_RATING_CATEGORY"),
        col("NUM_VOTES").alias("SR_NUM_VOTES")
    )
    
    sb_df = silver_basics.select(
        col("TCONST").alias("SB_TCONST"),
        col("RUNTIME_MINUTES").alias("SB_RUNTIME_MINUTES")
    )
    
    # Aggregate episodes with SE_ prefix (BATCH aggregation)
    episode_agg = (
        silver_episode
        .groupBy(
            col("PARENTTCONST").alias("SE_PARENTTCONST"),
            col("SEASON_NUMBER").alias("SE_SEASON_NUMBER")
        )
        .agg(count("TCONST").alias("SE_NO_OF_EPISODES"))
    )
    
    # Read dimension with prefix (always BATCH)
    dt_df = dlt.read("gold.dim_title_basics").select(
        col("TITLE_BASICS_SK"),
        col("TITLE_BASICS_TCONST").alias("DT_TCONST")
    )
    
    # Join: STREAMING (sr_df) with BATCH (all others)
    # This is allowed - stream-to-static joins are supported
    fact_df = (
        sr_df
        .join(dt_df, col("SR_TCONST") == col("DT_TCONST"), "inner")
        .join(sb_df, col("SR_TCONST") == col("SB_TCONST"), "left")
        .join(episode_agg, col("SR_TCONST") == col("SE_PARENTTCONST"), "left")
    )
    
    # Generate SK using SR_ prefixed columns
    fact_df = generate_sk_with_prefix(fact_df, "FS", ["SR_TCONST", "SE_SEASON_NUMBER"])
    
    return fact_df.select(
        col("SK").alias("TITLE_STATS_SK"),
        col("TITLE_BASICS_SK"),
        col("SR_AVERAGE_RATING").cast("decimal(3,1)").alias("AVERAGE_RATING"),
        col("SR_RATING_CATEGORY").alias("RATING_CATEGORY"),
        col("SR_NUM_VOTES").cast("int").alias("NUM_VOTES"),
        col("SB_RUNTIME_MINUTES").cast("int").alias("RUNTIME_MINUTES"),
        coalesce(col("SE_SEASON_NUMBER").cast("int"), lit(-1)).alias("SEASON_NUMBER"),
        coalesce(col("SE_NO_OF_EPISODES").cast("int"), lit(-1)).alias("NO_OF_EPISODES"),
        lit("PRATHUSH").alias("DI_JOB_ID"),
        current_date().alias("DI_LOAD_DATE")
    )